# Detección de roya del cafeto (*Hemileia vastatrix*) y estimación de severidad
### Transfer learning (EfficientNetB0 / ResNet50) con el dataset RoCoLe

**Objetivo.** Entrenar una red neuronal convolucional que, a partir de la foto de una hoja de café:
1. Determine si **tiene roya** (sí / no).
2. Asigne el **nivel de severidad** según la escala OIRSA usada en RoCoLe.

| Clase RoCoLe | Significado | Área foliar afectada |
|---|---|---|
| `healthy` | Hoja sana | 0 % |
| `red_spider_mite` | Ácaro rojo (no es roya) | — |
| `rust_level_1` | Roya leve | 1–5 % |
| `rust_level_2` | Roya moderada | 6–20 % |
| `rust_level_3` | Roya severa | 21–50 % |
| `rust_level_4` | Roya muy severa | > 50 % |

**Dataset:** Parraga-Alava, J., Cusme, K., Loor, A., & Santander, E. (2019). *RoCoLe: A robusta coffee leaf images dataset for evaluation of machine learning based methods in plant diseases recognition*. Data in Brief, 25, 104414. https://doi.org/10.1016/j.dib.2019.104414 — Datos: https://doi.org/10.17632/c5yvn32dzg.2 (1 560 imágenes, licencia CC BY 4.0).

**Flujo:** descarga → lectura de etiquetas → partición estratificada → aumento de datos → entrenamiento en 2 fases (cabeza + *fine-tuning*) → evaluación (matriz de confusión, F1, Grad-CAM) → predicción de hojas nuevas → exportación a **TensorFlow Lite** para la app Android.

> Recomendado: ejecutar en **Google Colab con GPU** (Entorno de ejecución → Cambiar tipo → GPU T4).

## 1. Instalación e importaciones

In [ ]:
# %pip instala en el mismo Python que usa el kernel del cuaderno (local o Colab)
%pip install -q opencv-python openpyxl seaborn scikit-learn pandas matplotlib tensorflow

import os, glob, json, zipfile, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import cv2

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Obtener el dataset RoCoLe

**Opción A (recomendada):** entrar a https://data.mendeley.com/datasets/c5yvn32dzg/2 → *Download All* → subir el `.zip` a Colab o a Google Drive.

**Opción B:** si lo tienes en Kaggle, usa la API de Kaggle (`kaggle datasets download ...`) y apunta `ZIP_PATH` al archivo descargado.

Ajusta las rutas de la siguiente celda. El código busca automáticamente las fotos y el archivo de anotaciones dentro del zip.

In [ ]:
# --- Si usas Google Drive, descomenta: ---
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/datasets/RoCoLe.zip'

ZIP_PATH = 'RoCoLe.zip'          # ruta al zip descargado de Mendeley
DATA_DIR = 'rocole'              # carpeta de extracción

if not os.path.isdir(DATA_DIR):
    assert os.path.exists(ZIP_PATH), f"No encuentro {ZIP_PATH}. Sube el zip de RoCoLe."
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    # Mendeley a veces anida otros zips: los extraemos también
    for inner in glob.glob(f'{DATA_DIR}/**/*.zip', recursive=True):
        with zipfile.ZipFile(inner) as z:
            z.extractall(os.path.splitext(inner)[0])

imgs = [p for p in glob.glob(f'{DATA_DIR}/**/*', recursive=True)
        if p.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Imágenes encontradas: {len(imgs)}")

## 3. Leer las etiquetas

RoCoLe incluye una hoja de anotaciones (`RoCoLe-classes.xlsx` / `.csv`) con el nombre del archivo, la etiqueta binaria (*healthy / unhealthy*) y la multiclase (*healthy, red_spider_mite, rust_level_1…4*). Si tu copia viene organizada en **carpetas por clase** (versiones de Kaggle), el código también lo detecta.

In [ ]:
CLASSES = ['healthy', 'red_spider_mite', 'rust_level_1', 'rust_level_2', 'rust_level_3', 'rust_level_4']
name2path = {os.path.basename(p).lower(): p for p in imgs}

def normalizar(lbl):
    lbl = str(lbl).strip().lower().replace(' ', '_').replace('-', '_')
    for c in CLASSES:
        if c in lbl or lbl in c:
            return c
    if lbl.startswith('rust') and lbl[-1].isdigit():
        return f'rust_level_{lbl[-1]}'
    return None

def cargar_desde_anotaciones():
    cands = [p for p in glob.glob(f'{DATA_DIR}/**/*', recursive=True)
             if p.lower().endswith(('.xlsx', '.xls', '.csv'))]
    for f in cands:
        df = pd.read_excel(f) if f.lower().endswith(('xlsx', 'xls')) else pd.read_csv(f)
        cols = {c.lower(): c for c in df.columns}
        fcol = next((cols[c] for c in cols if 'file' in c or 'image' in c or 'name' in c), None)
        mcol = next((cols[c] for c in cols if 'multi' in c), None)
        if fcol and mcol:
            out = pd.DataFrame({
                'file': df[fcol].astype(str).map(lambda s: os.path.basename(s).lower()),
                'label': df[mcol].map(normalizar)})
            out['path'] = out['file'].map(name2path)
            out = out.dropna(subset=['path', 'label'])
            if len(out):
                print('Anotaciones leídas de:', f)
                return out[['path', 'label']]
    return None

def cargar_desde_carpetas():
    rows = [(p, normalizar(os.path.basename(os.path.dirname(p)))) for p in imgs]
    df = pd.DataFrame(rows, columns=['path', 'label']).dropna()
    return df if len(df) else None

df = cargar_desde_anotaciones()
if df is None:
    df = cargar_desde_carpetas()
assert df is not None and len(df), "No pude asociar imágenes con etiquetas; revisa la estructura del zip."

df['tiene_roya'] = df['label'].str.startswith('rust').astype(int)
print(df['label'].value_counts().reindex(CLASSES))

fig, ax = plt.subplots(figsize=(8, 3.5))
df['label'].value_counts().reindex(CLASSES).plot.bar(ax=ax, color='#6b8e23')
ax.set_title('Distribución de clases en RoCoLe'); ax.set_ylabel('Imágenes')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

### 3.1 Muestras visuales

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for j, c in enumerate(CLASSES):
    muestras = df[df.label == c].sample(min(2, (df.label == c).sum()), random_state=SEED)
    for i, (_, r) in enumerate(muestras.iterrows()):
        axes[i, j].imshow(plt.imread(r.path)); axes[i, j].set_title(c, fontsize=10)
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 4. Partición estratificada (70 / 15 / 15)

RoCoLe tiene **4 fotos por planta** (390 plantas). Si los nombres de archivo permiten identificar la planta, conviene separar por planta para evitar fuga de información; aquí usamos estratificación por clase, que es lo habitual en la literatura.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
for n, d in [('Entrenamiento', train_df), ('Validación', val_df), ('Prueba', test_df)]:
    print(f"{n:14s}: {len(d):4d}")

## 5. Pipeline de datos y aumento

- Tamaño de entrada 224 × 224 (estándar para EfficientNet/ResNet y ligero para el celular).
- Aumento: giros, rotación, zoom, brillo y contraste — simula condiciones de campo.
- **Pesos de clase** para compensar el desbalance (pocas hojas en nivel 3 y 4).

In [ ]:
IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE
cls2idx = {c: i for i, c in enumerate(CLASSES)}

def leer(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32), label

aumento = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='aumento')

def hacer_ds(d, entrenar=False):
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.label.map(cls2idx).values))
    if entrenar:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(leer, num_parallel_calls=AUTOTUNE)
    if entrenar:
        ds = ds.map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = hacer_ds(train_df, True), hacer_ds(val_df), hacer_ds(test_df)

pesos = compute_class_weight('balanced', classes=np.arange(len(CLASSES)),
                             y=train_df.label.map(cls2idx).values)
class_weight = dict(enumerate(pesos))
print({CLASSES[k]: round(v, 2) for k, v in class_weight.items()})

## 6. Modelo: transfer learning

Por defecto **EfficientNetB0** (preciso y ligero para TFLite). Cambia `BACKBONE = 'resnet50'` para replicar la arquitectura ResNet-50 de la tesis.

In [ ]:
BACKBONE = 'efficientnet'   # 'efficientnet' o 'resnet50'

def construir_modelo():
    entrada = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    if BACKBONE == 'resnet50':
        base = keras.applications.ResNet50(include_top=False, weights='imagenet')
        x = keras.applications.resnet50.preprocess_input(entrada)
    else:
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet')
        x = entrada   # EfficientNet de Keras ya incluye su normalización
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(len(CLASSES), activation='softmax', name='severidad')(x)
    return keras.Model(entrada, salida), base

model, base = construir_modelo()
model.summary(show_trainable=True)

### 6.1 Fase 1 — entrenar solo la cabeza clasificadora

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
      keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=cb)

### 6.2 Fase 2 — *fine-tuning* de las últimas capas del backbone

In [ ]:
base.trainable = True
for capa in base.layers[:-40]:          # descongelar solo las últimas 40 capas
    capa.trainable = False
for capa in base.layers:                 # BatchNorm congelado = fine-tuning estable
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb.append(keras.callbacks.ModelCheckpoint('mejor_modelo_roya.keras', save_best_only=True,
                                          monitor='val_loss'))
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=25,
                  class_weight=class_weight, callbacks=cb)

In [ ]:
def curvas(*hs):
    acc = sum([h.history['accuracy'] for h in hs], []); vacc = sum([h.history['val_accuracy'] for h in hs], [])
    loss = sum([h.history['loss'] for h in hs], []); vloss = sum([h.history['val_loss'] for h in hs], [])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(acc, label='entrenamiento'); ax[0].plot(vacc, label='validación'); ax[0].set_title('Exactitud')
    ax[1].plot(loss, label='entrenamiento'); ax[1].plot(vloss, label='validación'); ax[1].set_title('Pérdida')
    for a in ax:
        a.axvline(len(hs[0].history['loss']) - 0.5, ls='--', c='gray'); a.legend(); a.set_xlabel('Época')
    plt.tight_layout(); plt.show()
curvas(hist1, hist2)

## 7. Evaluación en el conjunto de prueba

Se evalúan dos tareas a partir de la misma salida:
- **Binaria** (¿roya sí/no?): suma de probabilidades de `rust_level_1…4`.
- **Severidad** (6 clases).

In [ ]:
y_true = test_df.label.map(cls2idx).values
probs = model.predict(test_ds)
y_pred = probs.argmax(1)

print('=== Clasificación por severidad (6 clases) ===')
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='YlGn', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title('Matriz de confusión normalizada')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

rust_idx = [cls2idx[c] for c in CLASSES if c.startswith('rust')]
p_roya = probs[:, rust_idx].sum(1)
yb_true = np.isin(y_true, rust_idx).astype(int)
yb_pred = (p_roya >= 0.5).astype(int)
print('=== Detección binaria de roya ===')
print(classification_report(yb_true, yb_pred, target_names=['sin roya', 'con roya'], digits=3))

# Error ordinal: qué tan lejos queda la severidad predicha de la real (solo hojas con roya)
mask = np.isin(y_true, rust_idx) & np.isin(y_pred, rust_idx)
if mask.any():
    nivel = lambda i: int(CLASSES[i][-1])
    err = np.abs(np.vectorize(nivel)(y_true[mask]) - np.vectorize(nivel)(y_pred[mask]))
    print(f'Error medio absoluto de nivel (hojas con roya): {err.mean():.2f} | '
          f'aciertos exactos o a ±1 nivel: {(err <= 1).mean():.1%}')

## 8. Explicabilidad — Grad-CAM

Muestra **dónde mira** la red: en una hoja con roya, el mapa de calor debería concentrarse en las pústulas anaranjadas.

In [ ]:
def gradcam(img_array):
    base_m = model.get_layer(base.name)
    with tf.GradientTape() as tape:
        x = img_array
        if BACKBONE == 'resnet50':
            x = keras.applications.resnet50.preprocess_input(x)
        fmap = base_m(x, training=False)
        tape.watch(fmap)
        pooled = model.layers[-3](fmap)          # GlobalAveragePooling
        pred = model.layers[-1](pooled)
        top = pred[:, tf.argmax(pred[0])]
    grads = tape.gradient(top, fmap)
    w = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fmap * w[:, None, None, :], -1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]

ejemplos = test_df[test_df.tiene_roya == 1].sample(4, random_state=SEED)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for a, (_, r) in zip(axes, ejemplos.iterrows()):
    img, _ = leer(r.path, 0)
    a.imshow(img.numpy().astype('uint8')); a.imshow(gradcam(img[None]), cmap='jet', alpha=0.4)
    a.set_title(f'Real: {r.label}'); a.axis('off')
plt.tight_layout(); plt.show()

## 9. Diagnosticar una hoja nueva

Sube una foto de una hoja (por ejemplo, de una huerta de Atoyac de Álvarez) y obtén diagnóstico, nivel de severidad y probabilidades.

In [ ]:
ETIQUETAS_ES = {
    'healthy': ('Sana', 'Sin roya', '0 %'),
    'red_spider_mite': ('Ácaro rojo', 'Sin roya (otra afectación)', '—'),
    'rust_level_1': ('Roya', 'Nivel 1 — leve', '1–5 %'),
    'rust_level_2': ('Roya', 'Nivel 2 — moderada', '6–20 %'),
    'rust_level_3': ('Roya', 'Nivel 3 — severa', '21–50 %'),
    'rust_level_4': ('Roya', 'Nivel 4 — muy severa', '> 50 %'),
}

def diagnosticar(ruta, mostrar=True):
    img, _ = leer(ruta, 0)
    p = model.predict(img[None], verbose=0)[0]
    c = CLASSES[p.argmax()]
    p_roya = p[rust_idx].sum()
    estado, severidad, area = ETIQUETAS_ES[c]
    res = {'archivo': os.path.basename(ruta),
           'tiene_roya': bool(p_roya >= 0.5),
           'prob_roya': round(float(p_roya), 3),
           'diagnostico': estado, 'severidad': severidad, 'area_afectada': area,
           'confianza': round(float(p.max()), 3)}
    if mostrar:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1, 1.3]})
        ax[0].imshow(img.numpy().astype('uint8')); ax[0].axis('off')
        ax[0].set_title(f"{estado} · {severidad}\n(área afectada {area})")
        ax[1].barh([ETIQUETAS_ES[k][1] if k.startswith('rust') else ETIQUETAS_ES[k][0] for k in CLASSES],
                   p, color=['#4caf50', '#9e9e9e', '#ffc107', '#ff9800', '#f4511e', '#b71c1c'])
        ax[1].set_xlim(0, 1); ax[1].set_title('Probabilidades')
        plt.tight_layout(); plt.show()
    return res

# Ejemplo con una imagen del conjunto de prueba:
print(diagnosticar(test_df.sample(1, random_state=1).path.iloc[0]))

# Para tus propias fotos en Colab:
# from google.colab import files
# for nombre in files.upload(): print(diagnosticar(nombre))

### 9.1 (Opcional) Estimación continua del % de área con roya por color

Complemento al clasificador: segmenta la hoja (verde) y las lesiones (amarillo-anaranjado) en HSV y calcula **S (%) = área lesionada / área foliar × 100**. Es una aproximación sensible a la iluminación; úsala como apoyo, no como verdad de campo.

In [ ]:
def severidad_por_color(ruta):
    bgr = cv2.resize(cv2.imread(ruta), (512, 512))
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    hoja = cv2.inRange(hsv, (10, 40, 40), (90, 255, 255))          # verde + amarillo + naranja
    hoja = cv2.morphologyEx(hoja, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    lesion = cv2.inRange(hsv, (5, 90, 90), (28, 255, 255)) & hoja   # tonos de pústula de roya
    s = 100 * lesion.sum() / max(hoja.sum(), 1)
    nivel = 0 if s < 1 else 1 if s <= 5 else 2 if s <= 20 else 3 if s <= 50 else 4
    return round(s, 2), nivel

r = test_df[test_df.tiene_roya == 1].sample(1, random_state=3).iloc[0]
print('Etiqueta real:', r.label, '| % estimado y nivel OIRSA:', severidad_por_color(r.path))

## 10. Guardar y exportar a TensorFlow Lite (app Android)

Se exporta el modelo con cuantización de rango dinámico (≈ 4× más pequeño) y un archivo `labels.txt` en el mismo orden de las salidas. La entrada es `float32 [1, 224, 224, 3]` con valores 0–255 (el preprocesamiento va dentro del modelo).

In [ ]:
model.save('modelo_roya_rocole.keras')

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()
open('modelo_roya.tflite', 'wb').write(tflite)
open('labels.txt', 'w').write('\n'.join(CLASSES))
print(f"TFLite: {len(tflite)/1e6:.1f} MB")

# Verificación rápida: el TFLite debe dar la misma clase que Keras
interp = tf.lite.Interpreter(model_content=tflite); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
img, _ = leer(test_df.path.iloc[0], 0)
interp.set_tensor(inp['index'], img[None].numpy()); interp.invoke()
print('Keras:', CLASSES[model.predict(img[None], verbose=0).argmax()],
      '| TFLite:', CLASSES[interp.get_tensor(out['index']).argmax()])

# En Colab, para descargar:
# from google.colab import files
# files.download('modelo_roya.tflite'); files.download('labels.txt')

## 11. Notas metodológicas

- **Desbalance:** los niveles 3 y 4 tienen muy pocas imágenes; además de los pesos de clase, puede probarse fusionar niveles (p. ej. *leve* = 1, *moderada* = 2, *severa* = 3–4) y reportar ambas versiones.
- **Validez externa:** RoCoLe es *Coffea canephora* (robusta) en Ecuador. Para Arabica en Guerrero conviene validar con fotos locales o complementar con **BRACOL** (Arabica, Brasil; https://doi.org/10.17632/yy2k5y8mxg.1), que también trae severidad.
- **Validación cruzada:** para la tesis, repetir con *k*-fold estratificado (k = 5) y reportar media ± DE de exactitud y F1-macro.
- **Métricas recomendadas:** F1-macro (por el desbalance), matriz de confusión y error ordinal de nivel.